# 04 - HTR baseline: CNN + sequence encoder + CTC

Goal: build the model, prove the plumbing works, and measure size and speed **before** touching a real dataset.

Pipeline: `line image -> CNN (vision) -> sequence encoder (BiLSTM or Transformer) -> CTC head -> greedy decode`

Nothing here needs data. Real training starts in notebook 05 once a dataset is chosen.

In [1]:
import io, sys, time
from pathlib import Path

import torch
from torch import nn

# Find the repo root (the folder that contains training/models) so imports work from any cwd.
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "training" / "models").is_dir())
sys.path.insert(0, str(ROOT))

from training.models.charset import CharTokenizer
from training.models.crnn import HTRConfig, HTRModel, count_parameters
from training.models.ctc_utils import greedy_decode, cer, wer

torch.manual_seed(0)
print("repo root:", ROOT, "| torch", torch.__version__)

repo root: /workspace | torch 2.4.0


## 1. Tokenizer

CTC needs a blank symbol. Id 0 is always the blank; real characters start at 1. In notebook 01 the character set will be built from the **training** transcriptions only.

In [2]:
sample_texts = ["the cat sat", "neural nets", "hello world", "gradient"]
tok = CharTokenizer.from_texts(sample_texts)
print("characters:", "".join(tok.chars))
print("num_classes (incl. blank):", tok.num_classes)

ids = tok.encode("neural")
print(ids, "->", tok.decode(ids))

characters:  acdeghilnorstuw
num_classes (incl. blank): 17
[10, 5, 15, 12, 2, 9] -> neural


## 2. Build both model variants

Same CNN and CTC head; only the sequence encoder changes. This gives a controlled comparison for notebook 06:
a recurrent encoder (BiLSTM) versus self-attention (Transformer encoder, the building block of NLP models).

In [3]:
models = {}
for enc in ("bilstm", "transformer"):
    cfg = HTRConfig(num_classes=tok.num_classes, encoder=enc)
    models[enc] = HTRModel(cfg)
    print(f"{enc:12s} {count_parameters(models[enc]) / 1e6:.2f}M parameters")

x = torch.randn(2, 1, 64, 256)  # (batch, channel, height, width)
for enc, m in models.items():
    m.eval()
    with torch.no_grad():
        logits = m(x)
    print(enc, "input", tuple(x.shape), "-> logits", tuple(logits.shape), "(batch, time steps, classes)")

bilstm       2.18M parameters
transformer  2.97M parameters
bilstm input (2, 1, 64, 256) -> logits (2, 64, 17) (batch, time steps, classes)
transformer input (2, 1, 64, 256) -> logits (2, 64, 17) (batch, time steps, classes)


The width is downsampled by 4, so a 256 px wide line gives 64 time steps. CTC needs at least as many time steps as characters (plus one extra step between repeated letters), so check this against the longest line once the dataset is chosen.

## 3. CTC loss: forward and backward

`CTCLoss` wants log-probabilities shaped `(T, B, C)`, the output lengths, and the targets with their lengths.

In [4]:
targets = [torch.tensor(tok.encode(t)) for t in sample_texts]
target_cat = torch.cat(targets)
target_len = torch.tensor([len(t) for t in targets])

images = torch.randn(4, 1, 64, 192)
widths = torch.full((4,), 192)
ctc = nn.CTCLoss(blank=0, zero_infinity=True)

m = models["bilstm"].train()
logits = m(images, widths)
log_probs = logits.log_softmax(-1).permute(1, 0, 2)  # (T, B, C)
loss = ctc(log_probs, target_cat, m.output_lengths(widths), target_len)
loss.backward()
print("loss:", round(loss.item(), 3))
print("untrained decode:", [tok.decode(i) for i in greedy_decode(logits.detach())])

loss: 9.97
untrained decode: ['ststcsstsc', 'tstscstscscscstcsic', 'tscscssctstc', 'tstsasascsists']


## 4. Overfit sanity test

A correct pipeline must be able to memorise 4 samples. If the loss does not fall towards 0 here, something is wrong with shapes, lengths, the blank index or the loss setup, and no amount of real data would fix it.

This only proves the plumbing. It says nothing about generalisation; that is what the validation CER in notebook 05 is for.

In [ ]:
def overfit(encoder: str, steps: int = 250):
    torch.manual_seed(0)
    model = HTRModel(HTRConfig(num_classes=tok.num_classes, encoder=encoder, dropout=0.0)).train()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for step in range(1, steps + 1):
        logits = model(images, widths)
        lp = logits.log_softmax(-1).permute(1, 0, 2)
        loss = ctc(lp, target_cat, model.output_lengths(widths), target_len)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        if step in (1, 50, 100, 200, steps):
            print(f"  {encoder} step {step:3d} loss {loss.item():.4f}")
    model.eval()
    with torch.no_grad():
        return [tok.decode(i) for i in greedy_decode(model(images))]

for enc in ("bilstm", "transformer"):
    hyp = overfit(enc)
    print(enc, "->", hyp, "| CER", round(cer(sample_texts, hyp), 3))

  bilstm step   1 loss 9.9668
  bilstm step  50 loss 2.6845
  bilstm step 100 loss 1.9556


## 5. Size and speed

The project targets CPU laptops and Android, so these are tracked from day one: parameters, fp32 size, and latency for one line at batch size 1 (the application's real inference setting).
These numbers come from your machine; re-measure on the target device later, and again after ONNX export and quantisation (notebook 08).

In [ ]:
x1 = torch.randn(1, 1, 64, 512)
for enc, m in models.items():
    m.eval()
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    with torch.inference_mode():
        for _ in range(3):
            m(x1)
        t0 = time.perf_counter()
        for _ in range(20):
            m(x1)
        ms = (time.perf_counter() - t0) / 20 * 1000
    print(f"{enc:12s} {count_parameters(m) / 1e6:.2f}M params | {buf.getbuffer().nbytes / 1e6:.1f} MB fp32 | {ms:.0f} ms per 64x512 line")

## 6. Metrics check

CER and WER are corpus-level: total edits divided by total reference characters (or words).

In [ ]:
refs = ["the convolutional neural network extracts features"]
hyps = ["the convolutlonal neural netwrok extracts feautures"]
print("CER", round(cer(refs, hyps), 3), "| WER", round(wer(refs, hyps), 3))

## Next

1. Notebook 01: pick the dataset, build the real character set, check the longest line against the time-step budget.
2. Notebook 05: real training loop with validation CER/WER, checkpoints and TensorBoard.
3. Notebook 06: compare `bilstm` vs `transformer` under identical data, augmentation and seed.